In [36]:
import pandas as pd
data=pd.read_csv('sdu_course_data.csv')
data

,student_id,faculty,major,current_semester,gpa,interest_area,elective_code,elective_name,elective_term,course_difficulty,ects,success_probability,passed,gpa_risk
0,201419610,School of Education and Humanities,Physics,4,2.28,Research,PHY310,Quantum Mechanics,5,Easy,5,0.39,0,High
1,202571945,School of IT and Applied Mathematics,Statistics and Data Science,1,3.12,"Research, Law, Programming",SDS306,Simulation Modeling,5,Medium,5,0.52,1,Low
2,203678638,School of Education and Humanities,Physics,5,2.31,"Artificial Intelligence, Data Analysis",PHY330,Electrodynamics,6,Medium,3,0.40,1,High
3,225437923,School of IT and Applied Mathematics,Information Systems,2,3.95,"Artificial Intelligence, Research",INF320,Information Security,6,Medium,5,0.64,1,Low
4,224226067,School of Education and Humanities,Mathematics,4,3.55,Programming,MAT310,Abstract Algebra,5,Easy,4,0.58,1,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,204832090,School of IT and Applied Mathematics,Statistics and Data Science,5,2.68,"Law, Education, Machine Learning",SDS305,Applied Statistics,5,Hard,3,0.45,1,Medium
49996,211858975,School of Education and Humanities,Physics,4,2.93,"Data Analysis, Statistics",PHY410,Statistical Physics,7,Easy,5,0.49,0,Medium
49997,215881204,School of Education and Humanities,Pedagogy and Psychology,4,2.50,Finance,EDU210,Educational Psychology,3,Medium,4,0.53,1,Medium
49998,218742689,School of Education and Humanities,Physics,4,3.66,"Programming, Education, Data Analysis",PHY330,Electrodynamics,6,Medium,3,0.60,1,Low


In [37]:
data = data.drop(columns=['student_id', 'faculty', 'elective_name','current_semester'])

In [38]:
difficulty_map = {
    "Easy": 0,
    "Medium": 1,
    "Hard": 2
}

data['course_difficulty'] = data['course_difficulty'].map(difficulty_map)

In [39]:
data = pd.get_dummies(data, columns=['major'], drop_first=True, dtype=int)

In [40]:
data['interest_area'] = data['interest_area'].str.split(', ')

In [41]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
interest_encoded = pd.DataFrame(
    mlb.fit_transform(data['interest_area']),
    columns=[f"interest_{c}" for c in mlb.classes_],
    index=data.index
)

data = pd.concat([data.drop(columns=['interest_area']), interest_encoded], axis=1)

In [42]:
data['gpa_risk'] = data['gpa_risk'].map({
    "Low": 0,
    "Medium": 1,
    "High": 2
})

In [43]:
data = pd.get_dummies(data, columns=['elective_code'], drop_first=True, dtype=int)

In [44]:
data

,gpa,elective_term,course_difficulty,ects,success_probability,passed,gpa_risk,major_Economics,major_Finance,major_Information Systems,...,elective_code_MAT330,elective_code_MAT410,elective_code_MGT210,elective_code_MGT320,elective_code_MGT410,elective_code_PHY310,elective_code_PHY330,elective_code_PHY410,elective_code_SDS305,elective_code_SDS306
0,2.28,5,0,5,0.39,0,2,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,3.12,5,1,5,0.52,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,2.31,6,1,3,0.40,1,2,0,0,0,...,0,0,0,0,0,0,1,0,0,0
3,3.95,6,1,5,0.64,1,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,3.55,5,0,4,0.58,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,2.68,5,2,3,0.45,1,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
49996,2.93,7,0,5,0.49,0,1,0,0,0,...,0,0,0,0,0,0,0,1,0,0
49997,2.50,3,1,4,0.53,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
49998,3.66,6,1,3,0.60,1,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [45]:
X = data.drop(columns=['passed'])
y = data['passed']

In [46]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [47]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

lr.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [48]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("F1:", f1_score(y_test, y_pred_lr))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))

Logistic Regression
Accuracy: 0.589
F1: 0.593069306930693
ROC-AUC: 0.6256851331726728


In [49]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)

In [50]:
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("Random Forest")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("F1:", f1_score(y_test, y_pred_rf))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))

Random Forest
Accuracy: 0.5567
F1: 0.5763975155279503
ROC-AUC: 0.5796630614148371


In [51]:
!pip install catboost

In [52]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    verbose=False,
    random_state=42
)

cb.fit(X_train, y_train)

In [53]:
y_pred_cb = cb.predict(X_test)
y_proba_cb = cb.predict_proba(X_test)[:, 1]

print("CatBoost")
print("Accuracy:", accuracy_score(y_test, y_pred_cb))
print("F1:", f1_score(y_test, y_pred_cb))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_cb))

CatBoost
Accuracy: 0.5857
F1: 0.6047132907165347
ROC-AUC: 0.624184466316391


## Диагностика: почему метрики невысокие?

Все три модели показали похожие, довольно скромные метрики (Accuracy ≈ 0.55–0.59, ROC-AUC ≈ 0.58–0.63). Прежде чем делать выводы, нужно проверить, в чём причина — в моделях или в самих данных. Проверю по порядку:

1. Баланс классов — не перекошена ли целевая переменная
2. Важность признаков — какие признаки реально влияют на предсказание
3. Train vs Test accuracy — нет ли переобучения/недообучения
4. Подбор гиперпараметров — можно ли улучшить результат тюнингом

In [54]:
#баланс классов 
print(data['passed'].value_counts(normalize=True))

passed
1    0.51522
0    0.48478
Name: proportion, dtype: float64


In [56]:
#важность признаков CatBoost
importance = pd.Series(cb.feature_importances_, index=X.columns).sort_values(ascending=False)
importance.head(10)

success_probability     27.597878
gpa                     11.908727
course_difficulty        5.730515
ects                     5.677274
elective_term            3.837242
interest_Statistics      3.471621
interest_Research        3.234590
interest_Education       2.913880
interest_Programming     2.697901
interest_Law             2.662600
dtype: float64

In [57]:
#проверка переобучения
y_train_pred_cb = cb.predict(X_train)
print("Train Accuracy:", accuracy_score(y_train, y_train_pred_cb))
print("Test Accuracy:", accuracy_score(y_test, y_pred_cb))

Train Accuracy: 0.6171
Test Accuracy: 0.5857


In [58]:
#подбор гиперпараметров
cb_tuned = CatBoostClassifier(
    iterations=800, depth=4, learning_rate=0.02, l2_leaf_reg=5,
    loss_function='Logloss', verbose=False, random_state=42
)
cb_tuned.fit(X_train, y_train)
y_pred_tuned = cb_tuned.predict(X_test)
y_proba_tuned = cb_tuned.predict_proba(X_test)[:, 1]

print("Tuned CatBoost")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned))
print("F1:", f1_score(y_test, y_pred_tuned))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_tuned))

Tuned CatBoost
Accuracy: 0.5882
F1: 0.6039623004423927
ROC-AUC: 0.6264168293770371


## Выводы

- **Баланс классов** близкий к равномерному (51.5% / 48.5%) — метрики не искажены дисбалансом.
- **Подбор гиперпараметров** (depth, learning_rate, регуляризация) практически не изменил метрики — прирост в пределах погрешности.
- **Важность признаков**: наиболее значимые — `success_probability` и `gpa`, но даже они слабо коррелируют с целевой переменной (`success_probability`: r≈0.28, `gpa`: r≈0.12).
- **Train vs Test accuracy** близки (0.62 vs 0.59) — модель не переобучена и не недообучена.

**Вывод**: потолок метрик (~0.59 Accuracy, ~0.62 ROC-AUC) определяется слабой предсказательной силой имеющихся признаков относительно целевой переменной, а не ограничениями модели или неудачным подбором гиперпараметров.